In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
from toponymy.annotation import NodeId, AnnotationTree, Annotation, AnnotationStore, Executor

In [7]:
import numpy as np

In [8]:
from toponymy.tools.notebook_data_load import load_small_newsgroups
newsgroups_df = load_small_newsgroups()
embeddings = np.stack(newsgroups_df["embedding"].values)
document_map = np.stack(newsgroups_df["map"].values)

In [9]:
from toponymy import ToponymyClusterer
clusterer = ToponymyClusterer(min_clusters=4, verbose=True)
clusterer.fit(document_map, embeddings);

Layer 0 found 8 clusters
Layer 1 found 4 clusters


In [10]:
clusterer.cluster_tree_

{(1, 0): [(0, 0)],
 (1, 1): [(0, 1)],
 (1, 2): [(0, 7), (0, 6), (0, 5)],
 (1, 3): [(0, 4)],
 (2, 0): [(1, 0), (1, 1), (1, 2), (1, 3), (0, 2), (0, 3)]}

## Nodes

In [11]:
node = NodeId(1, 0)

In [12]:
node.layer

1

In [13]:
node.cluster

0

In [14]:
node == (1, 0)

True

## AnnotationTree

Here are the assumptions about the condensed tree that must hold.

* There is exactly one root node.
* Every non-root node has exactly one parent.
* Edges may span more than one layer.
* The root is a catch-all node, and will not be considered a cluster node or a part of a layer.
* Layer ids are 0-indexed and there are no empty layers (this assumption is used by the executor)



Can make it from a condensed tree dict

In [15]:
tree = AnnotationTree(clusterer.cluster_tree_)

Or for an object with a `.cluster_tree_`

In [16]:
tree_from_clusterer = AnnotationTree.from_clusterer(clusterer)

Can check equality

In [17]:
tree == tree_from_clusterer

True

Can check if a node is in the tree

In [18]:
(0, 1) in tree

True

The root is not considered to be in the annotation tree

In [19]:
(2, 0) in tree

False

In [20]:
len(tree)

12

### Nodes
Has an iterable of all of the cluster nodes (the non-root nodes)

In [21]:
tree.nodes

(NodeId(0, 0),
 NodeId(0, 1),
 NodeId(0, 2),
 NodeId(0, 3),
 NodeId(0, 4),
 NodeId(0, 5),
 NodeId(0, 6),
 NodeId(0, 7),
 NodeId(1, 0),
 NodeId(1, 1),
 NodeId(1, 2),
 NodeId(1, 3))

In [22]:
len(tree) == len(tree.nodes)

True

### Layers
Gives access to the layers via their ids

In [23]:
tree.n_layers

2

In [24]:
tree.layer_ids

(0, 1)

In [25]:
tree.layer(0)

(NodeId(0, 0),
 NodeId(0, 1),
 NodeId(0, 2),
 NodeId(0, 3),
 NodeId(0, 4),
 NodeId(0, 5),
 NodeId(0, 6),
 NodeId(0, 7))

### Relatives

In [26]:
root_node = (2, 0)
leaf_node = (0, 0)
top_layer_node = (1, 2)

In [27]:
tree.parent(leaf_node)

NodeId(1, 0)

In [28]:
tree.parent(top_layer_node)

In [29]:
tree.children(leaf_node)

In [30]:
tree.children(top_layer_node)

[NodeId(0, 5), NodeId(0, 6), NodeId(0, 7)]

In [31]:
tree.descendants(top_layer_node)

[NodeId(0, 5), NodeId(0, 6), NodeId(0, 7)]

In [32]:
tree.descendants(leaf_node)

[]

The root is hidden from parent/child relationships since it doesn't represent a cluster

In [33]:
tree.parent(root_node)

In [34]:
tree.children(root_node)

In [35]:
tree.descendants(root_node)

[]

If you need to see the root or root children for some reason, there's internal access to them

In [36]:
tree._root

NodeId(2, 0)

In [37]:
tree._root_children

[NodeId(0, 2),
 NodeId(0, 3),
 NodeId(1, 0),
 NodeId(1, 1),
 NodeId(1, 2),
 NodeId(1, 3)]

## Annotation
An `Annotation` is a data store for annotations of the nodes of an `AnnotationTree`

It holds up to one object for each node in an `AnnotationTree`

In [38]:
example = Annotation("example", tree)

Here's an empty Annotation

In [39]:
example.states

{NodeId(0, 0): <AnnotationState.EMPTY: 'empty'>,
 NodeId(0, 1): <AnnotationState.EMPTY: 'empty'>,
 NodeId(0, 2): <AnnotationState.EMPTY: 'empty'>,
 NodeId(0, 3): <AnnotationState.EMPTY: 'empty'>,
 NodeId(0, 4): <AnnotationState.EMPTY: 'empty'>,
 NodeId(0, 5): <AnnotationState.EMPTY: 'empty'>,
 NodeId(0, 6): <AnnotationState.EMPTY: 'empty'>,
 NodeId(0, 7): <AnnotationState.EMPTY: 'empty'>,
 NodeId(1, 0): <AnnotationState.EMPTY: 'empty'>,
 NodeId(1, 1): <AnnotationState.EMPTY: 'empty'>,
 NodeId(1, 2): <AnnotationState.EMPTY: 'empty'>,
 NodeId(1, 3): <AnnotationState.EMPTY: 'empty'>}

In [40]:
example[leaf_node]

KeyError: NodeId(0, 0)

or the friendlier version:

In [41]:
example.get(leaf_node)

You can fill in an annotation from a layered list (aka. list of lists indexed by layer id and then cluster id)

In [42]:
# note, no need to cooerce the array to lists, except to be able to show equality below
centroid_vectors_layered_list = [list(clusterer.cluster_layers_[0].centroid_vectors), list(clusterer.cluster_layers_[1].centroid_vectors)]

In [43]:
centroid_annotation = Annotation.from_layered_list("centroids", tree, centroid_vectors_layered_list)

In [44]:
centroid_annotation.states

{NodeId(0, 0): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 1): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 2): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 3): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 4): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 5): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 6): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(0, 7): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(1, 0): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(1, 1): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(1, 2): <AnnotationState.COMPUTED: 'computed'>,
 NodeId(1, 3): <AnnotationState.COMPUTED: 'computed'>}

In [45]:
node = NodeId(1, 0)

In [46]:
all(clusterer.cluster_layers_[node.layer].centroid_vectors[node.cluster] == centroid_annotation[node])

True

And get a layered list of lists back from the annotation as long as all of the nodes have been computed. This gives backwards compatibility with the old representations. 

In [47]:
new_layered_list = centroid_annotation.to_layered_list()

In [48]:
centroid_vectors_layered_list == new_layered_list

True

## AnnotationStore

In [49]:
store = AnnotationStore(tree, [centroid_annotation])

In [50]:
store

AnnotationStore(centroids[12/12])

In [51]:
all(store['centroids'][node] == store.centroids[node])

True

In [52]:
all(store.centroids[node] == store.node(node)['centroids'])

True

In [53]:
store.node_states(node)

{'centroids': <AnnotationState.COMPUTED: 'computed'>}

## Annotator

In [54]:
class FirstEntryAnnotator:
    inputs = ("centroids",)
    outputs = ("first_item",)
    algorithm_type = "node-node"

    def annotate(
        self,
        node,
        *,
        centroids # has to match inputs name
    ):
        return {"first_item" : centroids[0]} # keys have to match outputs

## Executor

In [55]:
executor = Executor(store)

In [56]:
failures = executor.run(FirstEntryAnnotator())

In [57]:
failures

{}

In [58]:
store

AnnotationStore(centroids[12/12], first_item[12/12])

In [59]:
store['first_item'][node]

np.float64(-0.010361493309028446)

In [60]:
store['centroids'][node][0]

np.float64(-0.010361493309028446)